# Stage 1: Select Implied Copyable Trades

Find profitable follower BUYs that follow a leader's BUY or SELL on the same
token within a time window. Two separate leader groups: **buy leaders** (whose
BUYs precede follower BUYs) and **sell leaders** (whose SELLs precede follower
BUYs).

Grid-search over selection thresholds to maximize **copyable PnL from implied
trades** on the validation split.

**Output:** `stage1_implied_result.json` with best selection params.

In [27]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    select_follower_wallets,
    select_leader_wallets,
    detect_implied_buys,
    score_leaders,
    evaluate_implied_pnl,
    evaluate_follower_buy_performance,
    evaluate_leader_performance,
    evaluate_leader_followed_performance,
    iterative_leader_follower_filter,
    run_implied_grid_search,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data

In [28]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full)

Markets: 1877548
Filtered markets for {'Politics'}: 42733
Loading 16 trade shards...
Total trades loaded: 12,979,027
Unique wallets: 34,334
Date range: 2025-01-01 00:00:59+00:00 -> 2026-07-22 05:06:48+00:00
Data range:  2025-01-01 .. 2027-01-01
Train end:   2026-05-27
Val end:     2026-09-13

  Train:  9,816,433 trades  (14,473 markets)  .. 2026-05-27
  Val:    2,779,074 trades  (4,990 markets)  .. 2026-09-13
  Test:     383,520 trades  (  355 markets)  .. 2027-01-01
  Total: 12,979,027 trades  (19,818 markets)


## Compute wallet metrics on training data

In [29]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 33257


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,-0.1008,21.5751,1248.8660,4.4692,6
1,0x00090e8b4fa8f88dc9c1740e460dd0f670021d43,-0.0009,0.0248,-6145.7216,-0.0000,814
2,0x000b88e5ff8880d41f87070a7bd8bab414220872,0.3070,0.0306,272.3678,0.6899,55
3,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0.0215,0.3371,19375.3812,-0.0023,3509
4,0x000da5c4606f0c03bbe4dcbafc4458bdd10e54e0,0.1755,-0.0246,270.0859,0.0044,159
5,0x001f5386c206c14340f331aa2ce76e929cc29cb5,0.1001,-0.0100,1689.8660,-0.3593,116
6,0x0025222a9968b7620c4a28f01f77768af131644c,0.1069,-1.0000,75.7080,-0.0543,19
7,0x002a380091a15a37d0ea7e144db922b6b6899a00,0.1835,-0.1234,32.0975,-0.0030,44
8,0x002bb2d390146b98728c903b11b9526e54923ccf,0.7383,-0.0244,-37.3605,-0.0000,7
9,0x002dcd37b0b8fa8db98236e599fe1b90d6272561,0.0323,0.0650,3363.8995,0.0598,1125


## Baseline selection

In [30]:
follower_wallets = select_follower_wallets(
    wallet_vol,
    min_copyable_roi=0.05,
    min_trade_value=100,
    min_num_buckets=10,
    max_market_pnl_hhi=0.3,
)
print(f"Followers: {len(follower_wallets)}")

buy_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=20,
    min_roi=None,
    max_market_pnl_hhi=1,
    side="BUY",
)
print(f"Buy leaders: {len(buy_leaders)}")

sell_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=20,
    min_roi=None,
    max_market_pnl_hhi=1,
    side="SELL",
)
print(f"Sell leaders: {len(sell_leaders)}")

Followers: 1510
Buy leaders: 11897
Sell leaders: 11897


## Baseline evaluation

In [31]:
follower_ws = set(follower_wallets['wallet'])
buy_leader_ws = set(buy_leaders['wallet'])
sell_leader_ws = set(sell_leaders['wallet'])

for split_name, df_split in  [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_ev = evaluate_implied_pnl(
        df_split, follower_ws, buy_leader_ws,
        time_window_minutes=5, leader_side="BUY"
    )
    sell_ev = evaluate_implied_pnl(
        df_split, follower_ws, sell_leader_ws,
        time_window_minutes=5, leader_side="SELL",
    )
    total = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
    print(f"{split_name}: buy_pnl={buy_ev['followed_copyable_pnl']:.2f} ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)  "
          f"sell_pnl={sell_ev['followed_copyable_pnl']:.2f} ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)  "
          f"total={total:.2f}")

TRAIN: buy_pnl=6116632.14 (169953 trades, 8381 leaders)  sell_pnl=6909940.74 (209176 trades, 6548 leaders)  total=13026572.88
VAL: buy_pnl=1683895.15 (62572 trades, 2954 leaders)  sell_pnl=2129861.70 (74705 trades, 2168 leaders)  total=3813756.85
TEST: buy_pnl=-109218.00 (6626 trades, 1215 leaders)  sell_pnl=-172147.02 (7743 trades, 872 leaders)  total=-281365.02


## Score leaders (baseline)

In [32]:
# Buy leaders
buy_implied = detect_implied_buys(
    df_train, follower_ws, buy_leader_ws,
    time_window_minutes=5, leader_side="BUY",
)
buy_scores = score_leaders(buy_implied)
print("Top buy leaders:")
print(buy_scores.head(10).to_string())

print()

# Sell leaders
sell_implied = detect_implied_buys(
    df_train, follower_ws, sell_leader_ws,
    time_window_minutes=5, leader_side="SELL",
)
sell_scores = score_leaders(sell_implied)
print("Top sell leaders:")
print(sell_scores.head(10).to_string())

Top buy leaders:
                                leader_wallet  num_followers  total_follower_copyable_pnl  num_followed_trades  unique_tokens
0  0xe6c9eee27de792a9fdcdd322822452625d0f975e            471                  217114.7190                 2292            389
1  0x0e00ce71d933fd3ad48928911ef95c7327c8d1ff              2                  152532.7271                    5              1
2  0xf0b0ef1d6320c6be896b4c9c54dd74407e7f8cab            736                  138409.7038                 5023           1145
3  0x8f7a4b414417911e7e9bd738399874792cdbdb40            175                  124228.1079                  433            163
4  0x9ad091ca2e8f1bd69f27662edcb49dceeaa5bf3d            223                  121895.9507                  594            111
5  0x270fd599b1c57d04d893a3e799ed635254ca5e39             65                  110615.6339                  165              7
6  0xa3c2ec158f222f3f4f5f4f57b113d761a4d3df3d            293                  103443.0008            

## Grid search

Vary selection thresholds to maximize copyable PnL from implied trades on the
validation split.

In [39]:
# Params for Weather: 
# param_grid = dict(
#     # Follower selection
#     min_follower_copyable_roi=[0, 0.05],
#     min_follower_trade_value=[100],
#     min_follower_num_buckets=[10],
#     # Buy leader selection
#     min_buy_leader_trade_count=[20],
#     min_buy_leader_roi=[0.0],
#     max_buy_leader_hhi=[1],
#     # Sell leader selection
#     min_sell_leader_trade_count=[20],
#     min_sell_leader_roi=[-1],
#     max_sell_leader_hhi=[1],
#     # Detection
#     time_window_minutes=[10],
#     min_pair_interactions=[0],
#     # Iterative refinement
#     n_iterations=[2],
#     leader_min_copyable_pnl=[20.0],
#     follower_min_copyable_pnl=[20.0],
#     follower_min_copyable_roi=[0.05],
#     # Final cutoff
#     follower_min_copyable_roi_cutoff=[0.1], #[0.1, 0.15],
# )


# Params for Politics:
param_grid = dict(
    # Follower selection
    min_follower_copyable_roi=[0.02, 0.05],
    min_follower_trade_value=[1000],
    min_follower_num_buckets=[30],
    max_follower_hhi=[0.3],
    # Buy leader selection
    min_buy_leader_trade_count=[100],
    min_buy_leader_roi=[0.0],
    max_buy_leader_hhi=[1],
    # Sell leader selection
    min_sell_leader_trade_count=[100],
    min_sell_leader_roi=[-1],
    max_sell_leader_hhi=[1],
    # Detection
    time_window_minutes=[5],
    min_pair_interactions=[0],
    # Iterative refinement
    n_iterations=[3],
    leader_min_copyable_pnl=[None],
    follower_min_copyable_pnl=[100.0],
    follower_min_copyable_roi=[0.07],
    # Final cutoff
    follower_min_copyable_roi_cutoff=[0.1], #[0.1, 0.15],
)

n_combos = np.prod([len(v) for v in param_grid.values()])
print(f"Grid: {n_combos:.0f} combos")

Grid: 2 combos


In [40]:
res_df = run_implied_grid_search(param_grid, wallet_vol, df_train, df_val)
res_df.iloc[0]

Grid: 2 combos, 8 workers
  [2/2] 53.7s elapsed
Done: 2 configs in 54.2s


min_follower_copyable_roi                0.0200
min_follower_trade_value                   1000
min_follower_num_buckets                     30
max_follower_hhi                         0.3000
min_buy_leader_trade_count                  100
min_buy_leader_roi                       0.0000
max_buy_leader_hhi                            1
min_sell_leader_trade_count                 100
min_sell_leader_roi                          -1
max_sell_leader_hhi                           1
time_window_minutes                           5
min_pair_interactions                         0
n_iterations                                  3
leader_min_copyable_pnl                    None
follower_min_copyable_pnl              100.0000
follower_min_copyable_roi                0.0700
follower_min_copyable_roi_cutoff         0.1000
implied_copyable_pnl               2703856.2907
buy_pnl                            1175316.6595
sell_pnl                           1528539.6312
buy_trades                              

## Grid search results

In [41]:
best_row = res_df.iloc[0]
best_params = {k: best_row[k] for k in param_grid.keys()}

print(f"Best config (val implied_pnl={best_row['implied_copyable_pnl']:.2f}):")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"  followers={best_row['followers']:.0f}  buy_leaders={best_row['buy_leaders']:.0f}  sell_leaders={best_row['sell_leaders']:.0f}")
print(f"  buy_pnl={best_row['buy_pnl']:.2f}  sell_pnl={best_row['sell_pnl']:.2f}")

Best config (val implied_pnl=2703856.29):
  min_follower_copyable_roi: 0.02
  min_follower_trade_value: 1000
  min_follower_num_buckets: 30
  max_follower_hhi: 0.3
  min_buy_leader_trade_count: 100
  min_buy_leader_roi: 0.0
  max_buy_leader_hhi: 1
  min_sell_leader_trade_count: 100
  min_sell_leader_roi: -1
  max_sell_leader_hhi: 1
  time_window_minutes: 5
  min_pair_interactions: 0
  n_iterations: 3
  leader_min_copyable_pnl: None
  follower_min_copyable_pnl: 100.0
  follower_min_copyable_roi: 0.07
  follower_min_copyable_roi_cutoff: 0.1
  followers=224  buy_leaders=1281  sell_leaders=1198
  buy_pnl=1175316.66  sell_pnl=1528539.63


In [42]:
print("Top 10 results:")
res_df[["implied_copyable_pnl", "buy_pnl", "sell_pnl", "buy_trades", "sell_trades",
        "followers", "buy_leaders", "sell_leaders", "time_window_minutes"]].head(10)

Top 10 results:


,implied_copyable_pnl,buy_pnl,sell_pnl,buy_trades,sell_trades,followers,buy_leaders,sell_leaders,time_window_minutes
0,2703856.2907,1175316.6595,1528539.6312,16188,24472,224,1281,1198,5
1,2446387.9063,1061936.0534,1384451.8529,13555,20030,189,1204,1126,5


## Evaluate best config on all splits

In [43]:
# Re-select wallets with best params
best_followers = select_follower_wallets(
    wallet_vol,
    min_copyable_roi=best_params["min_follower_copyable_roi"],
    min_trade_value=best_params["min_follower_trade_value"],
    min_num_buckets=best_params["min_follower_num_buckets"],
    max_market_pnl_hhi=best_params["max_follower_hhi"],
)
best_buy_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=best_params["min_buy_leader_trade_count"],
    min_roi=best_params["min_buy_leader_roi"],
    max_market_pnl_hhi=best_params["max_buy_leader_hhi"],
    side="BUY",
)
best_sell_leaders = select_leader_wallets(
    wallet_vol,
    min_trade_count=best_params["min_sell_leader_trade_count"],
    min_roi=best_params["min_sell_leader_roi"],
    max_market_pnl_hhi=best_params["max_sell_leader_hhi"],
    side="SELL",
)

b_fw = set(best_followers["wallet"])
b_blw = set(best_buy_leaders["wallet"])
b_slw = set(best_sell_leaders["wallet"])
tw = best_params["time_window_minutes"]
min_pi = best_params["min_pair_interactions"]
n_iter = int(best_params.get("n_iterations", 0))

print(f"Initial: followers={len(b_fw)}  buy_leaders={len(b_blw)}  sell_leaders={len(b_slw)}")

# Iterative refinement on training data
if n_iter > 0:
    leader_pnl = best_params.get("leader_min_copyable_pnl")
    follower_pnl = best_params.get("follower_min_copyable_pnl", 20.0)
    follower_roi = best_params.get("follower_min_copyable_roi")
    b_fw, b_blw, b_slw, ref_log, ref_snaps = iterative_leader_follower_filter(
        df_train, b_fw, b_blw, b_slw,
        time_window_minutes=tw,
        n_iterations=n_iter,
        leader_min_copyable_pnl=leader_pnl,
        follower_min_copyable_pnl=follower_pnl,
        follower_min_copyable_roi=follower_roi,
    )

# Final follower cutoff by copyable_roi on training data
follower_roi_cutoff = best_params.get("follower_min_copyable_roi_cutoff")
if follower_roi_cutoff is not None:
    follower_pnl_thresh = best_params.get("follower_min_copyable_pnl", 20.0)
    buy_imp = detect_implied_buys(df_val, b_fw, b_blw, time_window_minutes=tw, leader_side="BUY")
    sell_imp = detect_implied_buys(df_val, b_fw, b_slw, time_window_minutes=tw, leader_side="SELL")
    combined = pd.concat([buy_imp, sell_imp], ignore_index=True)
    if not combined.empty:
        f_scores = combined.groupby("follower_wallet", sort=False).agg(
            total_copyable_pnl=("copyable_pnl", "sum"),
            total_copyable_notional=("copyable_notional", "sum"),
        ).reset_index()
        f_scores["copyable_roi"] = f_scores["total_copyable_pnl"] / f_scores["total_copyable_notional"].clip(lower=1e-9)
        b_fw = b_fw & set(f_scores.loc[
            (f_scores["total_copyable_pnl"] >= follower_pnl_thresh) & (f_scores["copyable_roi"] >= follower_roi_cutoff),
            "follower_wallet",
        ])

print(f"Final: followers={len(b_fw)}  buy_leaders={len(b_blw)}  sell_leaders={len(b_slw)}")

if n_iter > 0:
    print()
    print("Refinement convergence (train set):")
    for i, (fw_i, bl_i, sl_i) in enumerate(ref_snaps):
        buy_ev = evaluate_implied_pnl(df_train, fw_i, bl_i, time_window_minutes=tw, leader_side="BUY")
        sell_ev = evaluate_implied_pnl(df_train, fw_i, sl_i, time_window_minutes=tw, leader_side="SELL")
        cpnl = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
        cn = buy_ev["followed_copyable_notional"] + sell_ev["followed_copyable_notional"]
        croi = cpnl / cn if cn > 0 else 0.0
        print(f"  iter {i}: copyable_pnl={cpnl:>10.2f}  ROI={croi:>7.4f}  "
              f"followers={len(fw_i)}  buy_leaders={len(bl_i)}  sell_leaders={len(sl_i)}")


for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    buy_implied = detect_implied_buys(df_split, b_fw, b_blw, time_window_minutes=tw, leader_side="BUY")
    sell_implied = detect_implied_buys(df_split, b_fw, b_slw, time_window_minutes=tw, leader_side="SELL")
    buy_ev = evaluate_implied_pnl(df_split, b_fw, b_blw, time_window_minutes=tw, leader_side="BUY")
    sell_ev = evaluate_implied_pnl(df_split, b_fw, b_slw, time_window_minutes=tw, leader_side="SELL")
    follower_buy = evaluate_follower_buy_performance(df_split, b_fw)
    buy_leader_all = evaluate_leader_performance(df_split, b_blw, side="BUY")
    sell_leader_all = evaluate_leader_performance(df_split, b_slw, side="SELL")
    buy_leader_followed = evaluate_leader_followed_performance(df_split, buy_implied, b_blw, leader_side="BUY")
    sell_leader_followed = evaluate_leader_followed_performance(df_split, sell_implied, b_slw, leader_side="SELL")

    print(f"{split_name}:")
    # Following buy leaders
    b_roi = buy_ev["followed_copyable_pnl"] / buy_ev["followed_copyable_notional"] if buy_ev["followed_copyable_notional"] > 0 else 0.0
    print(f"  Following BUY leaders:")
    print(f"    Copyable PnL: {buy_ev['followed_copyable_pnl']:>10.2f}  ROI: {b_roi:>7.4f}  ({buy_ev['trade_count']} trades, {buy_ev['leader_count']} leaders)")
    print(f"    Wallet  PnL: {buy_ev['wallet_pnl']:>10.2f}  ROI: {buy_ev['wallet_roi']:>7.4f}")
    # Following sell leaders
    s_roi = sell_ev["followed_copyable_pnl"] / sell_ev["followed_copyable_notional"] if sell_ev["followed_copyable_notional"] > 0 else 0.0
    print(f"  Following SELL leaders:")
    print(f"    Copyable PnL: {sell_ev['followed_copyable_pnl']:>10.2f}  ROI: {s_roi:>7.4f}  ({sell_ev['trade_count']} trades, {sell_ev['leader_count']} leaders)")
    print(f"    Wallet  PnL: {sell_ev['wallet_pnl']:>10.2f}  ROI: {sell_ev['wallet_roi']:>7.4f}")
    # Combined
    imp_pnl = buy_ev["followed_copyable_pnl"] + sell_ev["followed_copyable_pnl"]
    imp_notional = buy_ev["followed_copyable_notional"] + sell_ev["followed_copyable_notional"]
    imp_trades = buy_ev["trade_count"] + sell_ev["trade_count"]
    imp_roi = imp_pnl / imp_notional if imp_notional > 0 else 0.0
    print(f"  Combined: Copyable PnL: {imp_pnl:>10.2f}  ROI: {imp_roi:>7.4f}  ({imp_trades} trades)")
    # All follower buys (regardless of leader)
    print(f"  All buys (followers):")
    print(f"    Copyable PnL: {follower_buy['followed_copyable_pnl']:>10.2f}  ROI: {follower_buy['followed_copyable_roi']:>7.4f}")
    print(f"    Wallet  PnL: {follower_buy['wallet_pnl']:>10.2f}  ROI: {follower_buy['wallet_roi']:>7.4f}  ({follower_buy['trade_count']} trades)")
    print(f"  Buy leaders (all buys):")
    print(f"    PnL: {buy_leader_all['pnl']:>10.2f}  ROI: {buy_leader_all['roi']:>7.4f}  ({buy_leader_all['trade_count']} trades)")
    print(f"  Buy leaders (followed buys):")
    print(f"    PnL: {buy_leader_followed['pnl']:>10.2f}  ROI: {buy_leader_followed['roi']:>7.4f}  ({buy_leader_followed['trade_count']} trades)")
    print(f"  Sell leaders (all sells):")
    print(f"    PnL: {sell_leader_all['pnl']:>10.2f}  ROI: {sell_leader_all['roi']:>7.4f}  ({sell_leader_all['trade_count']} trades)")
    print(f"  Sell leaders (followed sells):")
    print(f"    PnL: {sell_leader_followed['pnl']:>10.2f}  ROI: {sell_leader_followed['roi']:>7.4f}  ({sell_leader_followed['trade_count']} trades)")
    print()


Initial: followers=1403  buy_leaders=4087  sell_leaders=5042
Final: followers=224  buy_leaders=4087  sell_leaders=5042

Refinement convergence (train set):
  iter 0: copyable_pnl=15002292.86  ROI= 0.1700  followers=1403  buy_leaders=4087  sell_leaders=5042
  iter 1: copyable_pnl=14055585.12  ROI= 0.3275  followers=875  buy_leaders=4087  sell_leaders=5042
  iter 2: copyable_pnl=14055585.12  ROI= 0.3275  followers=875  buy_leaders=4087  sell_leaders=5042
  iter 3: copyable_pnl=14055585.12  ROI= 0.3275  followers=875  buy_leaders=4087  sell_leaders=5042
TRAIN:
  Following BUY leaders:
    Copyable PnL: 4022592.00  ROI:  0.3382  (54329 trades, 3084 leaders)
    Wallet  PnL: 7191699.69  ROI:  0.2476
  Following SELL leaders:
    Copyable PnL: 4799370.38  ROI:  0.3278  (77548 trades, 3128 leaders)
    Wallet  PnL: 8416596.91  ROI:  0.2575
  Combined: Copyable PnL: 8821962.37  ROI:  0.3325  (131877 trades)
  All buys (followers):
    Copyable PnL: 5245957.07  ROI:  0.2679
    Wallet  PnL: 119

## Save stage 1 result

In [44]:
import json
from datetime import datetime, timezone
from pathlib import Path


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


# Collect wallet records for each group
wallet_cols = [
    "wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "market_pnl_hhi",
]

def _wallet_records(df):
    if df is None or df.empty:
        return []
    cols = [c for c in wallet_cols if c in df.columns]
    records = df[cols].to_dict(orient="records")
    return [{k: _convert(v) for k, v in w.items()} for w in records]


metadata = {
    "type": "implied",
    "tags": ["Weather"],
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_followers": len(b_fw),
    "n_buy_leaders": len(b_blw),
    "n_sell_leaders": len(b_slw),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "type": "implied",
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_implied_copyable_pnl": float(best_row["implied_copyable_pnl"]),
    "metadata": metadata,
    "wallets": {
        "followers": _wallet_records(best_followers),
        "buy_leaders": _wallet_records(best_buy_leaders),
        "sell_leaders": _wallet_records(best_sell_leaders),
    },
}

out_path = Path("stage1_implied_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 implied result -> {out_path.resolve()}")


Saved stage 1 implied result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_implied_result.json
